# Validation #12: Predefined-Time Sliding Surface

## Reference
**Sánchez-Torres, J.D. et al. (2018).** "Predefined-time stability of dynamical systems with sliding modes."
*Mathematical Problems in Engineering*, 2018.

## OpenSMC Class
`PredefinedTimeSurface` in `+surfaces/PredefinedTimeSurface.m`

## The Surface

For $t < T_c$:
$$c(t) = \frac{\pi}{2T_c} \cdot \frac{1}{\cos\!\left(\frac{\pi}{2}\cdot\frac{t}{T_c}\right)}$$
$$s(t) = \dot{e} + c(t)\, e$$

For $t \geq T_c$:
$$s(t) = \dot{e} + c_\infty\, e \quad \text{(standard linear surface)}$$

**Key property:** The time-varying gain $c(t)$ grows to $\infty$ as $t \to T_c$,
forcing $e(t) \to 0$ **before** $T_c$ regardless of initial conditions.

## Plant
Double integrator: $\ddot{x} = u + d$

## Controller
$$u = -\dot{c}(t)\, e - c(t)\,\dot{e} - k\,\mathrm{sgn}(s) - \lambda\, s$$

where $\dot{c}(t) = \frac{\pi^2}{4T_c^2} \cdot \frac{\sin(\frac{\pi}{2}\frac{t}{T_c})}{\cos^2(\frac{\pi}{2}\frac{t}{T_c})}$ for $t < T_c$.

## Tests

| # | Test | Criterion |
|---|------|-----------|
| 1 | c(t) profile verification | c(0) = π/(2Tc), monotonic, c(t)→∞ |
| 2 | Convergence before Tc | |e(Tc)| < 0.01 for 9 configs |
| 3 | Linear vs Global vs PredefinedTime | Only PredefinedTime guarantees deadline |
| 4 | Robustness to disturbance | Convergence before Tc with d=2sin(3t) |
| 5 | Post-Tc behavior | Reverts to linear, stays regulated |
| 6 | Multiple Tc values | Convergence time scales with Tc |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'lines.linewidth': 1.5,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# ============================================================
# Predefined-Time Surface Functions
# ============================================================

def c_predefined(t, Tc):
    """Time-varying gain c(t) for t < Tc.
    c(t) = (pi / (2*Tc)) / cos((pi/2) * (t/Tc))
    """
    ratio = t / Tc
    ratio = min(ratio, 0.999)  # clamp to avoid numerical blow-up
    return (np.pi / (2.0 * Tc)) / np.cos((np.pi / 2.0) * ratio)


def cdot_predefined(t, Tc):
    """Time derivative of c(t) for t < Tc.
    dc/dt = (pi^2 / (4*Tc^2)) * sin((pi/2)*(t/Tc)) / cos^2((pi/2)*(t/Tc))
    """
    ratio = t / Tc
    ratio = min(ratio, 0.999)
    s = np.sin((np.pi / 2.0) * ratio)
    c = np.cos((np.pi / 2.0) * ratio)
    return (np.pi**2 / (4.0 * Tc**2)) * s / (c**2)


def surface_predefined(e, edot, t, Tc, c_inf=10.0):
    """Predefined-time sliding surface."""
    if t < Tc:
        ct = c_predefined(t, Tc)
    else:
        ct = c_inf
    return edot + ct * e


def simulate_predefined(Tc, e0, edot0=0.0, T=None, dt=1e-4, k=5.0,
                         lam=10.0, c_inf=10.0, disturbance=None):
    """Simulate double integrator with PredefinedTime surface.

    Plant: x_ddot = u + d
    Surface: s = edot + c(t)*e  (for t < Tc)
             s = edot + c_inf*e (for t >= Tc)
    Control: u = -cdot(t)*e - c(t)*edot - k*sign(s) - lam*s

    Uses RK4 integration.
    """
    if T is None:
        T = Tc * 3.0
    N = int(T / dt) + 1
    t_arr = np.linspace(0, T, N)

    e_arr = np.zeros(N)
    edot_arr = np.zeros(N)
    s_arr = np.zeros(N)
    u_arr = np.zeros(N)
    c_arr = np.zeros(N)

    x = np.array([e0, edot0])  # state = [e, edot]

    def get_control(t_i, e_i, edot_i):
        """Compute control input."""
        if t_i < Tc:
            ct = c_predefined(t_i, Tc)
            ct_dot = cdot_predefined(t_i, Tc)
        else:
            ct = c_inf
            ct_dot = 0.0
        s_i = edot_i + ct * e_i
        # u = -cdot*e - c*edot - k*sign(s) - lam*s
        u_i = -ct_dot * e_i - ct * edot_i - k * np.sign(s_i) - lam * s_i
        return u_i, s_i, ct

    def dynamics(t_i, state):
        """State derivative: [edot, e_ddot] where e_ddot = u + d."""
        e_i, edot_i = state
        u_i, _, _ = get_control(t_i, e_i, edot_i)
        d_i = disturbance(t_i) if disturbance is not None else 0.0
        return np.array([edot_i, u_i + d_i])

    for i in range(N):
        ti = t_arr[i]
        u_i, s_i, ct_i = get_control(ti, x[0], x[1])
        e_arr[i] = x[0]
        edot_arr[i] = x[1]
        s_arr[i] = s_i
        u_arr[i] = u_i
        c_arr[i] = ct_i

        if i < N - 1:
            # RK4
            h = dt
            k1 = dynamics(ti, x)
            k2 = dynamics(ti + h/2, x + h/2 * k1)
            k3 = dynamics(ti + h/2, x + h/2 * k2)
            k4 = dynamics(ti + h, x + h * k3)
            x = x + (h / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

    return {
        't': t_arr, 'e': e_arr, 'edot': edot_arr,
        's': s_arr, 'u': u_arr, 'c': c_arr,
        'Tc': Tc, 'e0': e0, 'c_inf': c_inf,
    }


print('Libraries and simulation functions loaded.')

## Test 1: c(t) Profile Verification

Verify that the time-varying gain $c(t)$ satisfies:
1. $c(0) = \frac{\pi}{2T_c}$
2. $c(t)$ is monotonically increasing on $[0, T_c)$
3. $c(t) \to \infty$ as $t \to T_c$

In [ ]:
# ============================================================
# TEST 1: c(t) profile for Tc = 2
# ============================================================

Tc_test = 2.0
N_pts = 10000
t_profile = np.linspace(0, Tc_test * 0.999, N_pts)
c_profile = np.array([c_predefined(ti, Tc_test) for ti in t_profile])

# Check 1: c(0) = pi / (2*Tc)
c0_theory = np.pi / (2.0 * Tc_test)
c0_actual = c_predefined(0.0, Tc_test)
c0_err = abs(c0_actual - c0_theory)
check1 = c0_err < 1e-12

# Check 2: Monotonically increasing
diffs = np.diff(c_profile)
check2 = np.all(diffs > 0)

# Check 3: c(t) -> infinity as t -> Tc
# Evaluate at t = 0.999*Tc
c_near_Tc = c_predefined(0.999 * Tc_test, Tc_test)
check3 = c_near_Tc > 100  # should be very large

# Also check at t = 0.9999*Tc (clamped to 0.999 in code)
ratio_999 = 0.999
c_analytical = (np.pi / (2.0 * Tc_test)) / np.cos((np.pi / 2.0) * ratio_999)

print('TEST 1: c(t) Profile Verification (Tc = 2.0)')
print('=' * 60)
print()
print(f'Check 1: c(0) = pi/(2*Tc) = {c0_theory:.6f}')
print(f'  Computed c(0) = {c0_actual:.6f}')
print(f'  Error: {c0_err:.2e}  -->  {"PASS" if check1 else "FAIL"}')
print()
print(f'Check 2: Monotonically increasing on [0, Tc)')
print(f'  All differences > 0: {check2}  -->  {"PASS" if check2 else "FAIL"}')
print(f'  Min diff = {np.min(diffs):.6e}')
print()
print(f'Check 3: c(t) -> infinity as t -> Tc')
print(f'  c(0.999*Tc) = {c_near_Tc:.2f}')
print(f'  c(0.99*Tc)  = {c_predefined(0.99*Tc_test, Tc_test):.2f}')
print(f'  c(0.9*Tc)   = {c_predefined(0.9*Tc_test, Tc_test):.2f}')
print(f'  c(0.5*Tc)   = {c_predefined(0.5*Tc_test, Tc_test):.2f}')
print(f'  Growth to {c_near_Tc:.1f} confirms divergence  -->  {"PASS" if check3 else "FAIL"}')
print()

all_pass = check1 and check2 and check3
print(f'TEST 1 RESULT: {"PASS" if all_pass else "FAIL"}')

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full profile
axes[0].plot(t_profile, c_profile, 'b-', linewidth=2)
axes[0].axhline(y=c0_theory, color='r', ls='--', alpha=0.5,
                label=f'c(0) = $\\pi/(2T_c)$ = {c0_theory:.4f}')
axes[0].axvline(x=Tc_test, color='k', ls=':', alpha=0.5, label=f'$T_c$ = {Tc_test}')
axes[0].set_xlabel('time (s)')
axes[0].set_ylabel('c(t)')
axes[0].set_title(f'Time-varying gain c(t) for $T_c$ = {Tc_test} s')
axes[0].legend()
axes[0].set_ylim([0, 50])

# Log scale to show divergence
axes[1].semilogy(t_profile, c_profile, 'b-', linewidth=2)
axes[1].axvline(x=Tc_test, color='k', ls=':', alpha=0.5, label=f'$T_c$ = {Tc_test}')
axes[1].set_xlabel('time (s)')
axes[1].set_ylabel('c(t)  [log scale]')
axes[1].set_title(f'c(t) divergence as t $\\to$ $T_c$')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig12_ct_profile.png', dpi=150)
plt.show()

## Test 2: Convergence Before Tc (Key Claim)

The defining property of the PredefinedTime surface: error converges to zero
**before** the user-specified time $T_c$, regardless of initial conditions.

We test 3 values of $T_c$ x 3 initial conditions = **9 configurations**.
All must satisfy $|e(T_c)| < 0.01$.

In [ ]:
# ============================================================
# TEST 2: Convergence before Tc for 9 configurations
# Tc in {1, 2, 5}, e0 in {1, 2, 5}
# Criterion: |e(Tc)| < 0.01
# ============================================================

Tc_vals = [1.0, 2.0, 5.0]
e0_vals = [1.0, 2.0, 5.0]

results_t2 = []
sims_t2 = {}

print('TEST 2: Convergence Before Tc')
print('=' * 75)
print(f'{"Tc":>6}  {"e(0)":>6}  {"e(Tc)":>12}  {"edot(Tc)":>12}  {"Status":>8}')
print('-' * 75)

for Tc in Tc_vals:
    for e0 in e0_vals:
        res = simulate_predefined(Tc=Tc, e0=e0, k=5.0, lam=10.0, c_inf=10.0)
        sims_t2[(Tc, e0)] = res

        # Find the index closest to t = Tc
        idx_Tc = np.argmin(np.abs(res['t'] - Tc))
        e_at_Tc = res['e'][idx_Tc]
        edot_at_Tc = res['edot'][idx_Tc]
        passed = abs(e_at_Tc) < 0.01
        results_t2.append((Tc, e0, e_at_Tc, edot_at_Tc, passed))

        status = 'PASS' if passed else 'FAIL'
        print(f'{Tc:6.1f}  {e0:6.1f}  {e_at_Tc:12.6f}  {edot_at_Tc:12.6f}  {status:>8}')

n_pass = sum(r[4] for r in results_t2)
print('-' * 75)
print(f'\nTEST 2 RESULT: {n_pass}/{len(results_t2)} PASS')
print(f'Overall: {"PASS" if n_pass == len(results_t2) else "FAIL"}')

# --- Plot: 3x3 grid ---
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for i, Tc in enumerate(Tc_vals):
    for j, e0 in enumerate(e0_vals):
        ax = axes[i, j]
        res = sims_t2[(Tc, e0)]
        ax.plot(res['t'], res['e'], 'b-', linewidth=1.5)
        ax.axvline(x=Tc, color='r', ls='--', alpha=0.7, label=f'$T_c$={Tc}')
        ax.axhline(y=0, color='k', ls='-', alpha=0.2)
        ax.axhline(y=0.01, color='g', ls=':', alpha=0.4)
        ax.axhline(y=-0.01, color='g', ls=':', alpha=0.4)
        idx_Tc = np.argmin(np.abs(res['t'] - Tc))
        ax.plot(Tc, res['e'][idx_Tc], 'ro', markersize=8)
        ax.set_title(f'$T_c$={Tc}, e(0)={e0}')
        ax.set_xlabel('time (s)')
        ax.set_ylabel('e(t)')
        ax.legend(fontsize=9)
        ax.set_xlim([0, res['t'][-1]])

plt.suptitle('Test 2: Convergence Before $T_c$ (red dot = e at $T_c$)', fontsize=14)
plt.tight_layout()
plt.savefig('fig12_convergence_before_Tc.png', dpi=150)
plt.show()

## Test 3: Comparison -- Linear vs Global vs PredefinedTime

Three surfaces on the same double integrator with the same initial conditions.
Only the PredefinedTime surface **guarantees** convergence by a specific deadline.

In [ ]:
# ============================================================
# TEST 3: Linear vs Global vs PredefinedTime comparison
# Same plant (double integrator), same IC (e0=15, edot0=0)
# All three use the SAME controller gains (k=5, lam=10).
# With large e0, linear/global surfaces still have significant
# error at t=Tc, while PredefinedTime is converged.
# ============================================================

dt = 1e-4
e0_cmp = 15.0
edot0_cmp = 0.0
Tc_cmp = 2.0
T_cmp = 8.0
N_cmp = int(T_cmp / dt) + 1
t_cmp = np.linspace(0, T_cmp, N_cmp)
c_lin = 3.0    # moderate surface slope for linear/global
k_sw = 5.0     # same switching gain for all
lam_sw = 10.0  # same smoothing gain for all


def sim_surface(surface_type):
    """Simulate one of three surface types."""
    x = np.array([e0_cmp, edot0_cmp])
    e_out = np.zeros(N_cmp)

    # For Global surface: record initial s
    s0_global = edot0_cmp + c_lin * e0_cmp
    alpha_global = 2.0

    for i in range(N_cmp):
        ti = t_cmp[i]
        ei, edi = x

        if surface_type == 'linear':
            s = edi + c_lin * ei
            u = -c_lin * edi - k_sw * np.sign(s) - lam_sw * s

        elif surface_type == 'global':
            s_nominal = edi + c_lin * ei
            s = s_nominal - s0_global * np.exp(-alpha_global * ti)
            # Equivalent control for global surface
            u = (-c_lin * edi
                 + alpha_global * s0_global * np.exp(-alpha_global * ti)
                 - k_sw * np.sign(s) - lam_sw * s)

        elif surface_type == 'predefined':
            if ti < Tc_cmp:
                ct = c_predefined(ti, Tc_cmp)
                ct_dot = cdot_predefined(ti, Tc_cmp)
            else:
                ct = 10.0
                ct_dot = 0.0
            s = edi + ct * ei
            u = -ct_dot * ei - ct * edi - k_sw * np.sign(s) - lam_sw * s

        e_out[i] = ei

        if i < N_cmp - 1:
            def dyn(t_i, state):
                return np.array([state[1], u])
            h = dt
            k1 = dyn(ti, x)
            k2 = dyn(ti + h/2, x + h/2 * k1)
            k3 = dyn(ti + h/2, x + h/2 * k2)
            k4 = dyn(ti + h, x + h * k3)
            x = x + (h / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

    return e_out


e_linear = sim_surface('linear')
e_global = sim_surface('global')
e_predef = sim_surface('predefined')

# Find convergence times (|e| < 0.01)
def find_conv_time(e_arr, t_arr, tol=0.01):
    for i in range(len(e_arr)):
        if np.all(np.abs(e_arr[i:min(i+1000, len(e_arr))]) < tol):
            return t_arr[i]
    return None

t_conv_lin = find_conv_time(e_linear, t_cmp)
t_conv_glo = find_conv_time(e_global, t_cmp)
t_conv_pre = find_conv_time(e_predef, t_cmp)

# Error magnitudes at t = Tc
idx_Tc = np.argmin(np.abs(t_cmp - Tc_cmp))
e_lin_at_Tc = abs(e_linear[idx_Tc])
e_glo_at_Tc = abs(e_global[idx_Tc])
e_pre_at_Tc = abs(e_predef[idx_Tc])

print('TEST 3: Surface Comparison (e0=15, Tc=2, k=5, lam=10)')
print('=' * 70)
print(f'{"Surface":<20} {"Conv. time":>12} {"Before Tc=2?":>14} {"|e(Tc)|":>12}')
print('-' * 70)

for name, t_c, e_Tc in [('Linear', t_conv_lin, e_lin_at_Tc),
                          ('Global', t_conv_glo, e_glo_at_Tc),
                          ('PredefinedTime', t_conv_pre, e_pre_at_Tc)]:
    t_str = f'{t_c:.4f} s' if t_c is not None else f'> {T_cmp} s'
    before = 'YES' if (t_c is not None and t_c < Tc_cmp) else 'NO'
    print(f'{name:<20} {t_str:>12} {before:>14} {e_Tc:12.6f}')

# Assertion 1: PredefinedTime has converged at Tc (|e(Tc)| < 0.01)
check_predef_conv = e_pre_at_Tc < 0.01

# Assertion 2: Linear and Global have NOT converged at Tc (|e(Tc)| > 0.01)
check_others_not = (e_lin_at_Tc > 0.01) and (e_glo_at_Tc > 0.01)

# Assertion 3: PredefinedTime has the smallest error at Tc
check_smallest = (e_pre_at_Tc < e_lin_at_Tc) and (e_pre_at_Tc < e_glo_at_Tc)

print()
print(f'Check 1 - PredefinedTime |e(Tc)| < 0.01: '
      f'{"PASS" if check_predef_conv else "FAIL"}')
print(f'Check 2 - Linear & Global |e(Tc)| > 0.01: '
      f'{"PASS" if check_others_not else "FAIL"}')
print(f'Check 3 - PredefinedTime has smallest |e(Tc)|: '
      f'{"PASS" if check_smallest else "FAIL"}')
print(f'  |e_predef(Tc)| = {e_pre_at_Tc:.6f}  vs  '
      f'|e_linear(Tc)| = {e_lin_at_Tc:.6f},  '
      f'|e_global(Tc)| = {e_glo_at_Tc:.6f}')

all_pass = check_predef_conv and check_others_not and check_smallest
print(f'\nTEST 3 RESULT: {"PASS" if all_pass else "FAIL"}')

# --- Plot ---
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t_cmp, e_linear, 'b-', linewidth=2, label='Linear surface')
ax.plot(t_cmp, e_global, 'g--', linewidth=2, label='Global surface')
ax.plot(t_cmp, e_predef, 'r-', linewidth=2.5, label='PredefinedTime surface')
ax.axvline(x=Tc_cmp, color='k', ls=':', linewidth=2, alpha=0.7,
           label=f'Deadline $T_c$ = {Tc_cmp} s')
ax.axhline(y=0, color='k', ls='-', alpha=0.2)
ax.fill_between([Tc_cmp - 0.05, Tc_cmp + 0.05], -1, 16,
                color='red', alpha=0.1)
ax.plot(Tc_cmp, e_linear[idx_Tc], 'bs', markersize=10, zorder=5)
ax.plot(Tc_cmp, e_global[idx_Tc], 'g^', markersize=10, zorder=5)
ax.plot(Tc_cmp, e_predef[idx_Tc], 'ro', markersize=10, zorder=5)
ax.set_xlabel('time (s)')
ax.set_ylabel('e(t)')
ax.set_title('Test 3: Linear vs Global vs PredefinedTime — '
             'Only PredefinedTime guarantees convergence by $T_c$')
ax.legend(fontsize=11)
ax.set_xlim([0, T_cmp])
ax.set_ylim([-1, e0_cmp + 1])
plt.tight_layout()
plt.savefig('fig12_surface_comparison.png', dpi=150)
plt.show()

## Test 4: Robustness to Disturbance

Add a matched disturbance $d(t) = 2\sin(3t)$ to the double integrator.
Verify that convergence still happens before $T_c$ (the switching gain $k$
must exceed the disturbance bound).

In [ ]:
# ============================================================
# TEST 4: Robustness to disturbance d = 2*sin(3*t)
# Need k > |d|_max = 2 for sliding to be maintained.
# We use k=5 (safely above 2).
# ============================================================

Tc_rob = 2.0
e0_rob = 3.0

def dist_fn(t):
    return 2.0 * np.sin(3.0 * t)

res_nodist = simulate_predefined(Tc=Tc_rob, e0=e0_rob, k=5.0, lam=10.0,
                                  disturbance=None)
res_dist = simulate_predefined(Tc=Tc_rob, e0=e0_rob, k=5.0, lam=10.0,
                                disturbance=dist_fn)

# Check convergence at Tc
idx_Tc_nd = np.argmin(np.abs(res_nodist['t'] - Tc_rob))
idx_Tc_d = np.argmin(np.abs(res_dist['t'] - Tc_rob))

e_at_Tc_nodist = res_nodist['e'][idx_Tc_nd]
e_at_Tc_dist = res_dist['e'][idx_Tc_d]

pass_nodist = abs(e_at_Tc_nodist) < 0.01
pass_dist = abs(e_at_Tc_dist) < 0.01

print('TEST 4: Robustness to Disturbance')
print('=' * 60)
print(f'Disturbance: d(t) = 2*sin(3t),  |d|_max = 2')
print(f'Switching gain: k = 5 > 2 = |d|_max')
print(f'Tc = {Tc_rob}, e(0) = {e0_rob}')
print()
print(f'Without disturbance: e(Tc) = {e_at_Tc_nodist:.6f}  -->  '
      f'{"PASS" if pass_nodist else "FAIL"}')
print(f'With disturbance:    e(Tc) = {e_at_Tc_dist:.6f}  -->  '
      f'{"PASS" if pass_dist else "FAIL"}')
print()
print(f'TEST 4 RESULT: {"PASS" if (pass_nodist and pass_dist) else "FAIL"}')

# --- Plot ---
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(res_nodist['t'], res_nodist['e'], 'b-', linewidth=1.5,
             label='No disturbance')
axes[0].plot(res_dist['t'], res_dist['e'], 'r--', linewidth=1.5,
             label='d(t) = 2sin(3t)')
axes[0].axvline(x=Tc_rob, color='k', ls=':', linewidth=2, alpha=0.7,
                label=f'$T_c$ = {Tc_rob}')
axes[0].axhline(y=0, color='k', ls='-', alpha=0.2)
axes[0].set_xlabel('time (s)')
axes[0].set_ylabel('e(t)')
axes[0].set_title('Test 4: Error convergence with and without disturbance')
axes[0].legend()
axes[0].set_xlim([0, res_dist['t'][-1]])

axes[1].plot(res_nodist['t'], res_nodist['s'], 'b-', linewidth=0.8,
             label='s(t) no disturbance')
axes[1].plot(res_dist['t'], res_dist['s'], 'r-', linewidth=0.8,
             label='s(t) with disturbance')
axes[1].axvline(x=Tc_rob, color='k', ls=':', linewidth=2, alpha=0.7)
axes[1].axhline(y=0, color='k', ls='-', alpha=0.2)
axes[1].set_xlabel('time (s)')
axes[1].set_ylabel('s(t)')
axes[1].set_title('Sliding variable comparison')
axes[1].legend()
axes[1].set_xlim([0, res_dist['t'][-1]])

plt.tight_layout()
plt.savefig('fig12_robustness.png', dpi=150)
plt.show()

## Test 5: Post-Tc Behavior

After $t \geq T_c$, the surface reverts to $s = \dot{e} + c_\infty e$ (standard linear).
Verify:
1. The gain $c(t)$ jumps to $c_\infty$ at $t = T_c$
2. The error stays near zero (regulation)
3. The sliding variable matches the linear surface formula

In [ ]:
# ============================================================
# TEST 5: Post-Tc behavior
# After Tc, surface should be s = edot + c_inf * e
# System should stay regulated near e = 0
# ============================================================

Tc_post = 2.0
c_inf_post = 10.0
T_post = 10.0  # long simulation to check regulation

res_post = simulate_predefined(Tc=Tc_post, e0=2.0, k=5.0, lam=10.0,
                                c_inf=c_inf_post, T=T_post)

# Check 1: Gain transitions to c_inf at Tc
idx_Tc = np.argmin(np.abs(res_post['t'] - Tc_post))
# Check a few points after Tc
post_Tc_mask = res_post['t'] >= Tc_post + 0.01  # small margin
c_post_vals = res_post['c'][post_Tc_mask]
check_gain = np.all(np.abs(c_post_vals - c_inf_post) < 1e-10)

# Check 2: Error stays small after Tc
e_post_vals = res_post['e'][post_Tc_mask]
max_e_post = np.max(np.abs(e_post_vals))
check_regulated = max_e_post < 0.05

# Check 3: Sliding variable matches linear formula
s_post_vals = res_post['s'][post_Tc_mask]
edot_post_vals = res_post['edot'][post_Tc_mask]
s_linear_check = edot_post_vals + c_inf_post * e_post_vals
s_match_err = np.max(np.abs(s_post_vals - s_linear_check))
check_linear = s_match_err < 1e-6

print('TEST 5: Post-Tc Behavior')
print('=' * 60)
print(f'Tc = {Tc_post}, c_inf = {c_inf_post}, T_sim = {T_post}')
print()
print(f'Check 1: Gain = c_inf after Tc')
print(f'  All c(t>Tc) == {c_inf_post}: {check_gain}  -->  '
      f'{"PASS" if check_gain else "FAIL"}')
print()
print(f'Check 2: Error stays regulated after Tc')
print(f'  max|e(t>Tc)| = {max_e_post:.6f}  -->  '
      f'{"PASS" if check_regulated else "FAIL"}')
print()
print(f'Check 3: s(t>Tc) = edot + c_inf*e (linear surface)')
print(f'  max|s - (edot + c_inf*e)| = {s_match_err:.2e}  -->  '
      f'{"PASS" if check_linear else "FAIL"}')
print()
all_post = check_gain and check_regulated and check_linear
print(f'TEST 5 RESULT: {"PASS" if all_post else "FAIL"}')

# --- Plot ---
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Gain c(t)
axes[0].plot(res_post['t'], res_post['c'], 'b-', linewidth=1.5)
axes[0].axvline(x=Tc_post, color='r', ls='--', alpha=0.7, label=f'$T_c$={Tc_post}')
axes[0].axhline(y=c_inf_post, color='g', ls=':', alpha=0.7,
                label=f'$c_\\infty$={c_inf_post}')
axes[0].set_ylabel('c(t)')
axes[0].set_title('Gain profile: time-varying before $T_c$, constant after')
axes[0].legend()
axes[0].set_ylim([0, 60])
axes[0].set_xlim([0, T_post])

# Error e(t)
axes[1].plot(res_post['t'], res_post['e'], 'b-', linewidth=1.5)
axes[1].axvline(x=Tc_post, color='r', ls='--', alpha=0.7)
axes[1].axhline(y=0, color='k', ls='-', alpha=0.2)
axes[1].set_ylabel('e(t)')
axes[1].set_title('Error: converges before $T_c$, stays regulated after')
axes[1].set_xlim([0, T_post])

# Sliding variable s(t)
axes[2].plot(res_post['t'], res_post['s'], 'b-', linewidth=0.8)
axes[2].axvline(x=Tc_post, color='r', ls='--', alpha=0.7)
axes[2].axhline(y=0, color='k', ls='-', alpha=0.2)
axes[2].set_xlabel('time (s)')
axes[2].set_ylabel('s(t)')
axes[2].set_title('Sliding variable: reaches and stays near zero')
axes[2].set_xlim([0, T_post])

plt.tight_layout()
plt.savefig('fig12_post_Tc.png', dpi=150)
plt.show()

## Test 6: Multiple Tc Values -- Convergence Time Scales with Tc

Verify that the convergence time scales proportionally with $T_c$.
For $T_c \in \{0.5, 1, 2, 5\}$, convergence should happen at approximately $T_c$.

In [ ]:
# ============================================================
# TEST 6: Multiple Tc values
# Tc in {0.5, 1, 2, 5}, same e0 = 2
# Verify convergence happens just before each Tc
# ============================================================

Tc_multi = [0.5, 1.0, 2.0, 5.0]
e0_multi = 2.0
colors = ['red', 'orange', 'blue', 'green']

results_t6 = []
sims_t6 = {}

print('TEST 6: Convergence Time Scales with Tc')
print('=' * 70)
print(f'{"Tc":>6}  {"e(Tc)":>12}  {"Conv time":>12}  {"Conv/Tc":>10}  {"Status":>8}')
print('-' * 70)

for Tc in Tc_multi:
    # Use higher k and lam for faster Tc to ensure convergence
    k_val = max(5.0, 10.0 / Tc)
    lam_val = max(10.0, 20.0 / Tc)
    res = simulate_predefined(Tc=Tc, e0=e0_multi, k=k_val, lam=lam_val,
                               c_inf=10.0, T=max(Tc*3, 3.0))
    sims_t6[Tc] = res

    idx_Tc = np.argmin(np.abs(res['t'] - Tc))
    e_at_Tc = res['e'][idx_Tc]

    # Find actual convergence time
    conv_time = None
    for i in range(len(res['e'])):
        if abs(res['e'][i]) < 0.01:
            # Check it stays small
            end_check = min(i + 1000, len(res['e']))
            if np.all(np.abs(res['e'][i:end_check]) < 0.05):
                conv_time = res['t'][i]
                break

    passed = abs(e_at_Tc) < 0.01
    ratio = conv_time / Tc if conv_time is not None else None
    ratio_str = f'{ratio:.4f}' if ratio is not None else 'N/A'
    conv_str = f'{conv_time:.4f} s' if conv_time is not None else 'N/A'

    results_t6.append((Tc, e_at_Tc, conv_time, ratio, passed))
    print(f'{Tc:6.1f}  {e_at_Tc:12.6f}  {conv_str:>12}  {ratio_str:>10}  '
          f'{"PASS" if passed else "FAIL":>8}')

n_pass = sum(r[4] for r in results_t6)
print('-' * 70)
print(f'\nAll converge before Tc: {n_pass}/{len(results_t6)}')
print(f'TEST 6 RESULT: {"PASS" if n_pass == len(results_t6) else "FAIL"}')

# --- Plot: Overlay all Tc ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Absolute time
for Tc, color in zip(Tc_multi, colors):
    res = sims_t6[Tc]
    axes[0].plot(res['t'], res['e'], color=color, linewidth=1.5,
                label=f'$T_c$={Tc}')
    axes[0].axvline(x=Tc, color=color, ls=':', alpha=0.5)

axes[0].axhline(y=0, color='k', ls='-', alpha=0.2)
axes[0].set_xlabel('time (s)')
axes[0].set_ylabel('e(t)')
axes[0].set_title('Error vs absolute time')
axes[0].legend()
axes[0].set_xlim([0, 6])

# Normalized time (t / Tc)
for Tc, color in zip(Tc_multi, colors):
    res = sims_t6[Tc]
    t_norm = res['t'] / Tc
    mask = t_norm <= 2.0
    axes[1].plot(t_norm[mask], res['e'][mask], color=color, linewidth=1.5,
                label=f'$T_c$={Tc}')

axes[1].axvline(x=1.0, color='k', ls='--', linewidth=2, alpha=0.7,
                label='$t/T_c = 1$ (deadline)')
axes[1].axhline(y=0, color='k', ls='-', alpha=0.2)
axes[1].set_xlabel('$t / T_c$ (normalized time)')
axes[1].set_ylabel('e(t)')
axes[1].set_title('Error vs normalized time -- curves should all reach 0 by $t/T_c=1$')
axes[1].legend()
axes[1].set_xlim([0, 2.0])

plt.suptitle('Test 6: Convergence Time Scales with $T_c$', fontsize=14)
plt.tight_layout()
plt.savefig('fig12_multiple_Tc.png', dpi=150)
plt.show()

## Conclusion

### Validation Summary: PredefinedTimeSurface

| Test | Description | Criterion | Status |
|------|-------------|-----------|--------|
| 1 | c(t) profile | c(0)=pi/(2Tc), monotonic, diverges | Run notebook |
| 2 | Convergence before Tc | |e(Tc)| < 0.01 for 9 configs | Run notebook |
| 3 | Linear vs Global vs PredefinedTime | Only PredefinedTime meets deadline | Run notebook |
| 4 | Robustness to d=2sin(3t) | Convergence still before Tc | Run notebook |
| 5 | Post-Tc behavior | Reverts to linear, stays regulated | Run notebook |
| 6 | Multiple Tc values | Convergence time scales with Tc | Run notebook |

### Key Findings

1. The time-varying gain $c(t) = \frac{\pi}{2T_c \cos(\frac{\pi}{2}\frac{t}{T_c})}$ 
   starts at $\frac{\pi}{2T_c}$ and diverges as $t \to T_c$, forcing the error to zero.

2. Unlike fixed-time or finite-time surfaces, the convergence time $T_c$ is **independent 
   of initial conditions** -- the user specifies it directly.

3. After $T_c$, the surface smoothly reverts to standard linear sliding, maintaining 
   regulation without the aggressive time-varying gain.

4. The controller requires $k > |d|_{\max}$ (standard SMC requirement) for robustness.

### OpenSMC Implementation Notes

- `PredefinedTimeSurface.m` clamps `ratio = min(ratio, 0.999)` to avoid `cos(pi/2) = 0` singularity
- Parameters: `Tc` (convergence time), `c_inf` (post-convergence slope)
- The `compute` method takes `(e, edot, ~, t)` -- the time argument is essential

### Citation
```
Sánchez-Torres, J.D. et al. (2018). "Predefined-time stability of 
dynamical systems with sliding modes." Mathematical Problems in 
Engineering, 2018, Article ID 3710532.
```